In [2]:
from pathlib import Path
import tkinter as tk
from tkinter import filedialog
import numpy as np
import pandas as pd

# --- STUDENT CONFIGURATION ---
STUDENT_ID = "TUPM-26-4014"
SEED_DIGIT = int(STUDENT_ID[-1])  # 4
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())  # 17
SAMPLES_PER_OUTCOME = 20 + ID_SUM  # 37 per class (Total balanced sample size = 74)

Path("output").mkdir(exist_ok=True)

# --- BULLETPROOF FILE SELECTOR DIALOG ---
print("Please select your 'secom.data' file in the popup window...")
root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)

data_file = filedialog.askopenfilename(
    title="Select secom.data", filetypes=[("Data files", "*.data"), ("All files", "*.*")]
)
if not data_file:
  raise ValueError("No data file selected!")

print(f"Please select your 'secom_labels.data' file in the popup window...")
labels_file = filedialog.askopenfilename(
    title="Select secom_labels.data",
    filetypes=[("Data files", "*.data"), ("All files", "*.*")],
)
if not labels_file:
  raise ValueError("No labels file selected!")

# --- LOAD DATA ---
X_raw = pd.read_csv(data_file, sep=r"\s+", header=None)
y_raw = pd.read_csv(labels_file, sep=r"\s+", header=None)
df = X_raw.copy()
df["Outcome"] = y_raw.iloc[:, 0]
X = df.iloc[:, :-1]

# 1. Selected Features (first 5 numeric columns)
selected_feature_indices = list(X.select_dtypes(include=[np.number]).columns[:5])

# 2. Balanced Sample Sampling (equal representation per outcome)
np.random.seed(SEED_DIGIT)
pass_group = df[df["Outcome"] == -1].sample(
    n=SAMPLES_PER_OUTCOME, random_state=SEED_DIGIT
)
fail_group = df[df["Outcome"] == 1].sample(
    n=SAMPLES_PER_OUTCOME, random_state=SEED_DIGIT
)
balanced_df = pd.concat([pass_group, fail_group]).reset_index(drop=True)
balanced_sample_size = int(len(balanced_df))  # Total size (e.g., 74)

# Impute missing values for the selected subset
subset_cols = selected_feature_indices + ["Outcome"]
imputed_df = balanced_df[subset_cols].fillna(
    balanced_df[selected_feature_indices].mean()
)

# 3 & 4. Absolute Standardized Mean Difference & Ranking via Pandas & NumPy
pass_subset = imputed_df[imputed_df["Outcome"] == -1][selected_feature_indices]
fail_subset = imputed_df[imputed_df["Outcome"] == 1][selected_feature_indices]

mean_diff = fail_subset.mean() - pass_subset.mean()
pooled_std = np.sqrt(
    (pass_subset.var(ddof=1) + fail_subset.var(ddof=1)) / 2
).replace(0, 1e-5)

abs_std_mean_diff = (mean_diff / pooled_std).abs()
highest_ranked_feature = int(abs_std_mean_diff.idxmax())
highest_abs_std_diff_val = float(abs_std_mean_diff.max())

# 5. NumPy-Pandas Statistics Match Verification Check
np_pass = np.nanmean(pass_subset.values, axis=0)
pd_pass = pass_subset.mean().values
numpy_pandas_match = bool(np.allclose(np_pass, pd_pass, equal_nan=True))

# Export required output files
balanced_df.to_csv("output/balanced_sample.csv", index=False)
abs_std_mean_diff.to_frame("Absolute_Standardized_Mean_Difference").to_csv(
    "output/feature_comparison.csv"
)

# --- CONCISE ASSESSMENT OUTPUT ---
print("\n--- Exercise 2 Assessment Data ---")
print(f"Balanced Sample Size: {balanced_sample_size}")
print(f"Selected Features: {selected_feature_indices}")
print(f"Highest-Ranked Feature: {highest_ranked_feature}")
print(
    f"Absolute Standardized Mean Difference:"
    f" {highest_abs_std_diff_val:.4f}"
)
print(f"NumPy-Pandas Statistics Match: {numpy_pandas_match}")

Please select your 'secom.data' file in the popup window...
Please select your 'secom_labels.data' file in the popup window...

--- Exercise 2 Assessment Data ---
Balanced Sample Size: 74
Selected Features: [0, 1, 2, 3, 4]
Highest-Ranked Feature: 4
Absolute Standardized Mean Difference: 0.1852
NumPy-Pandas Statistics Match: True
